In [1]:
from pyspark.sql import functions as F

In [2]:
import time
from spark_session import get_spark

In [4]:
import sources

In [5]:
spark = get_spark("perf")

In [8]:
def perform_eval():
    for n in [8, 200]:
        spark.conf.set("spark.sql.shuffle.partitions", n)
        per_customer = lake.groupBy("customer_id").agg(F.sum("quantity").alias("units"))
        t = time.perf_counter(); per_customer.write.mode("overwrite").parquet(f"data/demo/per_customer_{n}")
        print(f"shuffle partitions {n}: {per_customer.rdd.getNumPartitions()} partitions, {time.perf_counter() - t:.2}s")
        

In [ ]:
lake = spark.read.parquet("data/lake/orders")
print("partitions after read:", lake.rdd.getNumPartitions())

partitions after read: 8


In [13]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
perform_eval()

shuffle partitions 8: 8 partitions, 1.2s
shuffle partitions 200: 200 partitions, 1.6e+01s


In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
perform_eval()

shuffle partitions 8: 1 partitions, 0.78s
shuffle partitions 200: 1 partitions, 1e+01s


In [15]:
day = lake.where("order_date = '2026-10-03'")
print("one day:", day.rdd.getNumPartitions(), "repartition(4):", day.repartition(4).rdd.getNumPartitions(), "coalesce(1):",
      day.coalesce(1).rdd.getNumPartitions())

(lake.repartition("order_date")
 .write.mode("overwrite").partitionBy("order_date")
 .parquet("data/lake/orders_compact")
 )

one day: 8 repartition(4): 4 coalesce(1): 1
